# Solar — Unified Language Representation for MultiModal QA
### Yu et al., ACL 2023 (Findings) | T4 x2 Kaggle | No API Keys

## Architecture (Paper Figure 1 + Section 2)
```
INPUT: Question Q + {Text passages, Tables, Images}

STEP 1 — Unified Language Representation (Section 2.1)
  Table  → natural language template:
             'Row one's race is Santa Derby, the track is Santa Park.'
  Image  → Global: BLIP caption + image title
             Local:  object-attribute detection ('racing brown horse; white stadium')
             Combined: global; local  [spliced through punctuations]
  → All modalities become text clues in unified language space Z

STEP 2 — Retrieve (Section 2.2) — SKIPPED for MultimodalQA
  Paper: 'MMQA also provides a clue list, so retrieval is not necessary'
  K=30 clues would be retrieved by DPR (BERT dense retrieval)

STEP 3 — Rank (Section 2.2)
  BERT cross-encoder: si = Sigmoid(Linear(BERT(zi ⊕ q)))  [Equation 2]
  → Top-N=10 clues kept for generation

STEP 4 — Generate (Section 2.2)
  T5 encoder-decoder: top-N clues + question → answer
  Cross-modal reasoning via cross-attention in T5 decoding
```

**Key differences from UniMMQA:**
- Solar uses retrieval-based pipeline (retrieve→rank→generate)
- Table uses natural language sentences, NOT position tokens
- Image uses global+local strategies (not diversified sampling)
- BERT ranker filters irrelevant clues before T5 sees them

**Kaggle**: Accelerator = T4 x2, Internet = ON, same multimodalqa-dataset

In [ ]:
# ============================================================
# Cell 1 — Install dependencies
# ============================================================
import sys, subprocess

def run(cmd):
    print(">", " ".join(cmd) if isinstance(cmd, list) else cmd)
    subprocess.check_call(cmd, shell=isinstance(cmd, str))

run([sys.executable, "-m", "pip", "uninstall", "-y",
     "torch", "torchvision", "torchaudio", "pillow"])

run([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
])

run([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "pillow==11.3.0", "numpy<2.2",
    "transformers>=4.36.0", "accelerate>=0.26.0",
    "sentencepiece", "pandas", "tqdm", "tabulate",
    "requests", "easyocr",
])

print("\n✅ Done — restart kernel: Runtime → Restart Session → Run All")

## RESTART KERNEL AFTER CELL 1

In [ ]:
# ============================================================
# Cell 2 — Imports and configuration
# ============================================================
import sys, os, re, gc, json, time, random, shutil
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from collections import Counter

import requests
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
import torchvision

from transformers import (
    BlipProcessor, BlipForConditionalGeneration,
    DetrImageProcessor, DetrForObjectDetection,
    AutoTokenizer, AutoModelForSequenceClassification,
    T5Tokenizer, T5ForConditionalGeneration,
)
import importlib.metadata as importlib_metadata

print("✅ Imports successful")
print(f"PyTorch: {torch.__version__} | Torchvision: {torchvision.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    cap  = torch.cuda.get_device_capability(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | sm_{cap[0]}{cap[1]} | {vram:.1f}GB")

# ── Paper-exact model choices ─────────────────────────────────────────────────
# Paper Section 3.2: 'image captioning is generated by BLIP'
BLIP_MODEL  = "Salesforce/blip-image-captioning-large"
# Paper Section 3.2: 'image-attribute features obtained with VinVL'
# VinVL requires a custom repo; DETR is a practical open-source equivalent
DETR_MODEL  = "facebook/detr-resnet-50"
# Paper Section 3.2: 'backbone for retrieval and ranking is BERT'
BERT_MODEL  = "bert-base-uncased"
# Paper Section 3.2: 'generation model is based on T5'
T5_MODEL    = "allenai/unifiedqa-t5-large"

# ── Paper-exact pipeline settings ─────────────────────────────────────────────
K_RETRIEVE  = 30     # paper: K=30 retrieved clues (skipped for MMQA)
N_RANK      = 10     # paper: N=10 clues kept after ranking
MAX_INPUT_LEN  = 1024
MAX_OUTPUT_LEN = 128

# ── Runtime switches ──────────────────────────────────────────────────────────
RUN_N_EXAMPLES           = 20
SELECT_ONE_PER_TYPE      = False
USE_LOCAL_TEXTUALIZATION = True
RUN_BASELINES            = True
FETCH_WIKI_IMAGES        = True

OUTPUT_DIR  = Path("/kaggle/working/solar_outputs")
IMAGE_CACHE = OUTPUT_DIR / "image_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_CACHE.mkdir(parents=True, exist_ok=True)

print(f"\n✅ Config ready | K={K_RETRIEVE} retrieve (skipped) | N={N_RANK} rank")

In [ ]:
# ============================================================
# Cell 3 — GPU verification
# ============================================================
assert torch.cuda.is_available(), "No GPU"
for i in range(torch.cuda.device_count()):
    cap = torch.cuda.get_device_capability(i)
    assert cap[0] >= 7, f"Need T4 (sm_75+)"
x = torch.tensor([1, -2, 3]).cuda()
assert (x < 0).any().item()
print(f"✅ T4 GPU verified | torchvision {torchvision.__version__}")

In [ ]:
# ============================================================
# Cell 4 — Dataset path discovery (proven pattern)
# ============================================================
KAGGLE_INPUT_CANDIDATES = [
    Path("/kaggle/input/datasets/adibraian/multimodalqa-dataset"),
    Path("/kaggle/input/multimodalqa-dataset"),
    Path("/kaggle/input"),
]
EXPECTED_FILES = {
    "dev":    ["MMQA_dev.jsonl",    "dev.jsonl"],
    "texts":  ["MMQA_texts.jsonl",  "texts.jsonl"],
    "tables": ["MMQA_tables.jsonl", "tables.jsonl"],
    "images": ["MMQA_images.jsonl", "images.jsonl"],
}

def unwrap_kaggle_file(path: Path) -> Optional[Path]:
    if path.is_file(): return path
    if path.is_dir():
        files = [c for c in path.iterdir() if c.is_file()]
        if len(files) == 1: return files[0]
        for f in files:
            if f.suffix in ('.jsonl', '.json'): return f
    return None

def resolve_files():
    root = next((c for c in KAGGLE_INPUT_CANDIDATES if c.exists()), None)
    if not root: print("Dataset root not found"); return {}
    print(f"Root: {root}")
    resolved = {}
    for key, names in EXPECTED_FILES.items():
        found = next((unwrap_kaggle_file(root/n) for n in names if unwrap_kaggle_file(root/n)), None)
        resolved[key] = found
        print(f"  {key:8s}: {'OK: '+str(found) if found else 'NOT FOUND'}")
    return resolved

FILE_PATHS  = resolve_files()
DEV_FILE    = FILE_PATHS["dev"]
TEXTS_FILE  = FILE_PATHS["texts"]
TABLES_FILE = FILE_PATHS["tables"]
IMAGES_FILE = FILE_PATHS["images"]
assert all([DEV_FILE, TEXTS_FILE, TABLES_FILE, IMAGES_FILE])
print("All files resolved")

In [ ]:
# ============================================================
# Cell 5 — Load and index dataset
# ============================================================
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try: data.append(json.loads(line))
                except: pass
    return data

def parse_table_raw(data):
    tb      = data.get('table', {})
    headers = [h.get('column_name', '') for h in tb.get('header', [])]
    rows    = [[c.get('text', '') for c in row] for row in tb.get('table_rows', [])]
    return {"title": data.get('title', ''), "headers": headers, "rows": rows, "id": data.get('id', '')}

def parse_text(data):
    return {"title": data.get('title',''), "text": data.get('text',''), "id": data.get('id','')}

def parse_image(data):
    return {"title": data.get('title',''), "id": data.get('id','')}

print("Loading dataset (~30-60 seconds)...")
dev_data    = load_jsonl(DEV_FILE)
texts_raw   = load_jsonl(TEXTS_FILE)
tables_raw  = load_jsonl(TABLES_FILE)
images_raw  = load_jsonl(IMAGES_FILE)

text_lookup  = {e['id']: parse_text(e)      for e in texts_raw  if e.get('id')}
table_lookup = {e['id']: parse_table_raw(e) for e in tables_raw if e.get('id')}
image_lookup = {e['id']: parse_image(e)     for e in images_raw if e.get('id')}

print(f"✅ {len(dev_data):,} dev | {len(text_lookup):,} texts | {len(table_lookup):,} tables | {len(image_lookup):,} images")

In [ ]:
# ============================================================
# Cell 6 — STEP 1a: Solar Table Textualization (Section 2.1)
#
# Paper Figure 1:
#   'Row one's race is Santa Derby, the track is Santa Park.'
#   'Row two's race is Kentucky Derby, the track is Churchill Downs.'
#
# This is natural language — different from UniMMQA's
# position-enhanced format (header : h1 | h2 | row 1 : ...)
# ============================================================

NUMBER_WORDS = [
    'zero','one','two','three','four','five','six','seven','eight','nine',
    'ten','eleven','twelve','thirteen','fourteen','fifteen','sixteen',
    'seventeen','eighteen','nineteen','twenty'
]

def num_to_word(n: int) -> str:
    if 0 <= n < len(NUMBER_WORDS): return NUMBER_WORDS[n]
    return str(n)


def textualize_table(table_data: Optional[Dict]) -> str:
    """
    Solar Section 2.1 — Natural Language Table Textualization.
    Paper: 'arrange cells in a linear fashion'
    Format matches Figure 1: Row {word}'s {col} is {val}, the {col} is {val}.
    """
    if not table_data: return ""
    title   = table_data.get('title', '')
    headers = table_data.get('headers', [])
    rows    = table_data.get('rows', [])
    if not headers or not rows: return title

    sentences = []
    if title:
        sentences.append(f"The table is about {title}.")

    for row_idx, row in enumerate(rows, start=1):
        row_word = num_to_word(row_idx)
        parts = []
        for col, val in zip(headers, row):
            val, col = str(val).strip(), str(col).strip()
            if val and col and val not in ('', '-', '--', 'N/A'):
                parts.append((col, val))
        if not parts: continue
        first_col, first_val = parts[0]
        sentence = f"Row {row_word}'s {first_col} is {first_val}"
        for col, val in parts[1:]:
            sentence += f", the {col} is {val}"
        sentence += "."
        sentences.append(sentence)

    return " ".join(sentences)


# Demo — should match Figure 1
demo = {
    "title": "A.P. Warrior",
    "headers": ["Race", "Track"],
    "rows": [["Santa Derby", "Santa Park"], ["Kentucky Derby", "Churchill Downs"]]
}
print("Solar Table Textualization (Figure 1 style):")
print(textualize_table(demo))
print("✅ Table textualization ready")

In [ ]:
# ============================================================
# Cell 7 — Load models for image textualization
# Paper: Global = BLIP | Local = VinVL (we use DETR)
# ============================================================
gc.collect()
torch.cuda.empty_cache()

print(f"Loading BLIP (global): {BLIP_MODEL}")
blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL)
blip_model = BlipForConditionalGeneration.from_pretrained(
    BLIP_MODEL, torch_dtype=torch.float16, device_map="auto", use_safetensors=True
)
blip_model.eval()
print("✅ BLIP loaded")

if USE_LOCAL_TEXTUALIZATION:
    print(f"\nLoading DETR (local object detection): {DETR_MODEL}")
    detr_processor = DetrImageProcessor.from_pretrained(DETR_MODEL)
    detr_model = DetrForObjectDetection.from_pretrained(
        DETR_MODEL, torch_dtype=torch.float16
    ).to("cuda:0")
    detr_model.eval()
    print("✅ DETR loaded")
else:
    detr_processor = detr_model = None

gc.collect(); torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    u = torch.cuda.memory_allocated(i)/1e9; t = torch.cuda.get_device_properties(i).total_memory/1e9
    print(f"   GPU {i}: {u:.1f}/{t:.1f} GB")

In [ ]:
# ============================================================
# Cell 8 — Load BERT ranker + T5 generator
# ============================================================
import transformers.utils.import_utils as _iu
def _safe_noop(): pass
_iu.check_torch_load_is_safe = _safe_noop

print(f"Loading BERT ranker: {BERT_MODEL}")
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_ranker = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL, num_labels=1, ignore_mismatched_sizes=True,
).to("cuda:0")
bert_ranker.eval()
print("✅ BERT ranker loaded")

print(f"\nLoading T5 generator: {T5_MODEL}")
t5_tokenizer = T5Tokenizer.from_pretrained(T5_MODEL)
t5_model = T5ForConditionalGeneration.from_pretrained(
    T5_MODEL, torch_dtype=torch.float16, device_map="auto", use_safetensors=True
)
t5_model.eval()
print("✅ T5 generator loaded")

gc.collect(); torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    u = torch.cuda.memory_allocated(i)/1e9; t = torch.cuda.get_device_properties(i).total_memory/1e9
    print(f"   GPU {i}: {u:.1f}/{t:.1f} GB")

In [ ]:
# ============================================================
# Cell 9 — STEP 1b: Image Textualization (Solar Section 2.1)
#
# Paper Global: 'image caption model + image title'
# Paper Local:  'object-attribute detection model — racing brown horse'
# Paper combined: 'global and local sequences spliced through punctuations'
# ============================================================

WIKI_HEADERS = {"User-Agent": "Solar-Research/1.0 (academic research)"}

COCO_LABELS = [
    'N/A','person','bicycle','car','motorcycle','airplane','bus','train','truck',
    'boat','traffic light','fire hydrant','N/A','stop sign','parking meter','bench',
    'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
    'N/A','backpack','umbrella','N/A','N/A','handbag','tie','suitcase','frisbee',
    'skis','snowboard','sports ball','kite','baseball bat','baseball glove',
    'skateboard','surfboard','tennis racket','bottle','N/A','wine glass','cup',
    'fork','knife','spoon','bowl','banana','apple','sandwich','orange','broccoli',
    'carrot','hot dog','pizza','donut','cake','chair','couch','potted plant','bed',
    'N/A','dining table','N/A','N/A','toilet','N/A','tv','laptop','mouse','remote',
    'keyboard','cell phone','microwave','oven','toaster','sink','refrigerator','N/A',
    'book','clock','vase','scissors','teddy bear','hair drier','toothbrush'
]


def global_textualize(image: Image.Image, title: str = "") -> str:
    """Solar Global: BLIP caption + title, combined."""
    parts = []
    if title: parts.append(title)
    try:
        inputs = blip_processor(images=image, return_tensors="pt")
        inputs = {k: v.to(blip_model.device) for k, v in inputs.items()}
        with torch.no_grad():
            ids = blip_model.generate(**inputs, max_new_tokens=60, do_sample=False)
        cap = blip_processor.decode(ids[0], skip_special_tokens=True).strip()
        if cap: parts.append(cap)
    except: pass
    return "; ".join(parts)


def local_textualize(image: Image.Image, conf_threshold: float = 0.7) -> str:
    """Solar Local: object-attribute detection (DETR instead of VinVL)."""
    if not USE_LOCAL_TEXTUALIZATION or detr_model is None: return ""
    try:
        inputs = detr_processor(images=image, return_tensors="pt")
        inputs = {k: v.to(detr_model.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = detr_model(**inputs)
        target_sizes = torch.tensor([image.size[::-1]])
        results = detr_processor.post_process_object_detection(
            outputs, target_sizes=target_sizes, threshold=conf_threshold
        )[0]
        seen, phrases = set(), []
        for _, label_id in zip(results['scores'], results['labels']):
            label = COCO_LABELS[label_id.item()] if label_id.item() < len(COCO_LABELS) else 'object'
            if label != 'N/A' and label not in seen:
                seen.add(label); phrases.append(label)
            if len(phrases) >= 8: break
        return "; ".join(phrases)
    except: return ""


def textualize_image(image_path: Optional[str], title: str = "") -> str:
    """Solar combined image textualization: global; local."""
    if not image_path or not Path(image_path).exists():
        return title if title else ""
    try:
        image = Image.open(image_path).convert("RGB")
    except: return title if title else ""

    global_text = global_textualize(image, title)
    local_text  = local_textualize(image)

    if global_text and local_text: return f"{global_text}; {local_text}"
    return global_text or local_text or title or ""


print(f"✅ Image textualization ready")
print(f"   Global: BLIP caption + title")
print(f"   Local: DETR object detection ({'ON' if USE_LOCAL_TEXTUALIZATION else 'OFF'})")

In [ ]:
# ============================================================
# Cell 10 — STEP 3: BERT Cross-Encoder Ranking (Section 2.2)
#
# Paper Equation 2:
#   si = Sigmoid(Linear(BERT(zi ⊕ q)))
#
# For MultimodalQA: retrieval is skipped (MMQA provides clue list)
# We rank all available clues and keep top-N=10
# ============================================================

def rank_clues(question: str, clues: List[Dict], top_n: int = N_RANK) -> List[Dict]:
    """
    BERT cross-encoder ranking (Paper Equation 2).
    Input: question + clue text pairs
    Output: top-N clues sorted by sigmoid score
    """
    if not clues: return []

    scores = []
    batch_size = 8

    for i in range(0, len(clues), batch_size):
        batch = clues[i:i+batch_size]
        clue_texts = [c['text'][:400] for c in batch]

        # BERT input: [CLS] question [SEP] clue [SEP]
        enc = bert_tokenizer(
            [question] * len(clue_texts), clue_texts,
            padding=True, truncation=True, max_length=512, return_tensors='pt',
        ).to(bert_ranker.device)

        with torch.no_grad():
            logits = bert_ranker(**enc).logits.squeeze(-1)
            batch_scores = torch.sigmoid(logits).cpu().tolist()

        if isinstance(batch_scores, float): batch_scores = [batch_scores]
        scores.extend(batch_scores)

    for clue, score in zip(clues, scores):
        clue['rank_score'] = score

    ranked = sorted(clues, key=lambda c: c['rank_score'], reverse=True)
    return ranked[:top_n]


print("✅ BERT ranker ready")
print(f"   Equation 2: si = Sigmoid(Linear(BERT(zi + q)))")
print(f"   Top-N={N_RANK} clues passed to T5 generator")

In [ ]:
# ============================================================
# Cell 11 — STEP 4: T5 Generation (Section 2.2)
# ============================================================

def generate_answer(question: str, top_clues: List[Dict]) -> str:
    """
    Solar Section 2.2 Generation.
    Paper: top-N clues joined with q, passed to T5.
    'Through cross-attention in T5 decoding, reasoning between
    and within modalities can be naturally realized.'
    """
    if not top_clues: return ""
    clue_text = " ".join(c['text'] for c in top_clues)
    unified   = f"{question} \n {clue_text}"

    inputs = t5_tokenizer(
        unified, return_tensors="pt",
        max_length=MAX_INPUT_LEN, truncation=True,
    ).to(t5_model.device)

    with torch.no_grad():
        ids = t5_model.generate(
            **inputs, max_new_tokens=MAX_OUTPUT_LEN,
            num_beams=4, do_sample=False, early_stopping=True,
        )
    return t5_tokenizer.decode(ids[0], skip_special_tokens=True).strip()


def clean_answer(text: str) -> str:
    t = re.sub(r'^MMQA[:\s]*', '', text.strip(), flags=re.IGNORECASE).strip()
    m = re.search(
        r'(?:in the (?:film|movie)|called|titled|named|is|was|are)\s+"?([^".,]+)"?[.,\s]*$',
        t, re.IGNORECASE
    )
    if m:
        cand = m.group(1).strip().strip('"\'.,')
        if len(cand) < len(t) * 0.7 and len(cand) > 1: return cand
    return t


print("✅ T5 generator ready | Beam=4")

In [ ]:
# ============================================================
# Cell 12 — Build clue corpus + examples
# ============================================================

def fetch_wiki_image(title: str) -> Optional[str]:
    try:
        r = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params={"action":"query","titles":title,"prop":"pageimages",
                    "pithumbsize":400,"format":"json"},
            headers=WIKI_HEADERS, timeout=8
        )
        for p in r.json().get("query",{}).get("pages",{}).values():
            t = p.get("thumbnail",{}).get("source")
            if t: return t
    except: pass
    return None


def download_image(url: str, path: Path, retries: int = 3) -> Optional[Path]:
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=WIKI_HEADERS, timeout=15)
            if r.status_code == 200 and len(r.content) > 500:
                path.write_bytes(r.content)
                img = Image.open(path).convert("RGB")
                if img.size[0] > 0: return path
        except:
            if attempt < retries - 1: time.sleep(1)
        try: path.unlink(missing_ok=True)
        except: pass
    return None


def build_clue_corpus(text_ids, table_id, img_ids) -> List[Dict]:
    """Build the unified language clue space Z for a question."""
    clues = []

    for tid in text_ids:
        t = text_lookup.get(tid)
        if not t: continue
        ct = f"{t['title']}. {t['text']}".strip()
        if ct: clues.append({"text": ct, "source": "text", "id": tid})

    if table_id:
        tbl = table_lookup.get(table_id)
        if tbl:
            ct = textualize_table(tbl)
            if ct: clues.append({"text": ct, "source": "table", "id": table_id})

    for iid in img_ids:
        img = image_lookup.get(iid)
        if not img: continue
        cache = IMAGE_CACHE / f"{iid}.jpg"
        local = None
        if cache.exists():
            try: Image.open(cache).convert("RGB"); local = cache
            except: cache.unlink(missing_ok=True)
        if local is None and FETCH_WIKI_IMAGES:
            url = fetch_wiki_image(img['title'])
            if url: local = download_image(url, cache)
        img_text = textualize_image(str(local) if local else None, img['title'])
        if img_text:
            clues.append({"text": img_text, "source": "image", "id": iid,
                          "local_path": str(local) if local else None})

    return clues


def build_example(entry: Dict) -> Dict:
    meta = entry.get('metadata', {})
    ans_list = entry.get('answers', [])
    first = ans_list[0] if ans_list else {}
    gold  = first.get('answer', '') if isinstance(first, dict) else str(first)
    return {
        "id":       entry.get('qid', entry.get('id', '')),
        "question": entry.get('question', ''),
        "gold_answer": gold,
        "type":     meta.get('type', 'unknown'),
        "text_ids": meta.get('text_doc_ids', []),
        "table_id": meta.get('table_id'),
        "img_ids":  meta.get('image_doc_ids', []),
    }


sel_indices = list(range(min(RUN_N_EXAMPLES, len(dev_data))))
print(f"Building {len(sel_indices)} examples...")
examples = []
for i in tqdm(sel_indices):
    ex = build_example(dev_data[i])
    examples.append(ex)
    print(f"  [{ex['type']}] {ex['question'][:60]}")
    print(f"         Gold: {ex['gold_answer']}")

print(f"\n✅ {len(examples)} examples ready")

In [ ]:
# ============================================================
# Cell 13 — Full Solar Pipeline
# ============================================================

def run_solar(question, text_ids, table_id, img_ids, top_n=N_RANK, verbose=True) -> Dict:
    """
    Full Solar pipeline:
    Step 1: Textualize all modalities into clue corpus Z
    Step 2: Skip DPR (MMQA provides clue list pre-given)
    Step 3: BERT cross-encoder ranking — keep top-N
    Step 4: T5 generation from top-N clues
    """
    if verbose:
        print(f"\n{'='*65}")
        print(f"Q: {question}")
        print(f"{'='*65}")

    result = {"question": question}

    # Step 1: Build clue corpus
    if verbose: print("\n Step 1: Building unified clue corpus Z...")
    clues = build_clue_corpus(text_ids, table_id, img_ids)
    result['all_clues'] = clues
    if verbose:
        for c in clues: print(f"   [{c['source']}] {c['text'][:100]}")
        print(f"  {len(clues)} clues total")

    if not clues:
        result.update({'final_answer': '', 'raw_answer': '', 'ranked': []})
        return result

    # Step 2: Retrieval skipped for MMQA
    if verbose: print("\n Step 2: DPR retrieval SKIPPED (MMQA pre-provides clue list)")

    # Step 3: BERT ranking
    if verbose: print(f"\n Step 3: BERT cross-encoder ranking...")
    top_clues = rank_clues(question, clues, top_n=top_n)
    result['ranked'] = top_clues
    if verbose:
        for c in top_clues:
            print(f"   [{c['source']} s={c['rank_score']:.3f}] {c['text'][:80]}")

    # Step 4: T5 generation
    if verbose: print("\n Step 4: T5 generation...")
    raw_answer   = generate_answer(question, top_clues)
    final_answer = clean_answer(raw_answer)
    result.update({'raw_answer': raw_answer, 'final_answer': final_answer})

    if verbose:
        print(f"  Raw:   {raw_answer}")
        print(f"  Clean: {final_answer}")
    return result


print("✅ Full Solar pipeline ready")

In [ ]:
# ============================================================
# Cell 14 — Evaluation helpers
# ============================================================
def normalize(text):
    return re.sub(r'\W+', ' ', str(text).lower().strip()).split()

def exact_match(pred, gold):
    g = gold[0] if isinstance(gold, list) else str(gold)
    return normalize(pred) == normalize(g)

def token_f1(pred, gold):
    g = gold[0] if isinstance(gold, list) else str(gold)
    p, g = normalize(pred), normalize(g)
    if not p or not g: return 0.0
    common = Counter(p) & Counter(g)
    n = sum(common.values())
    if n == 0: return 0.0
    pr, rc = n/len(p), n/len(g)
    return 2*pr*rc/(pr+rc)

def evaluate(rows):
    em = [exact_match(r['prediction'], r['gold_answer']) for r in rows]
    f1 = [token_f1(r['prediction'],   r['gold_answer']) for r in rows]
    return {'EM%': round(sum(em)/len(em)*100,1) if em else 0,
            'F1%': round(sum(f1)/len(f1)*100,1) if f1 else 0, 'n': len(rows)}

def save_jsonl(rows, path):
    with open(path,'w',encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + '\n')

print("✅ Evaluation ready")

In [ ]:
# ============================================================
# Cell 15 — Run Solar
# ============================================================
solar_rows = []

for run_i, ex in enumerate(examples, 1):
    print(f"\n{'#'*65}")
    print(f"  Solar {run_i}/{len(examples)} | Type: {ex['type']}")
    print(f"{'#'*65}")

    result = run_solar(
        question=ex['question'], text_ids=ex['text_ids'],
        table_id=ex['table_id'], img_ids=ex['img_ids'],
        top_n=N_RANK, verbose=True,
    )

    ranked_sources = [c['source'] for c in result.get('ranked', [])]
    row = {
        "id": ex['id'], "type": ex['type'],
        "question": ex['question'], "gold_answer": ex['gold_answer'],
        "prediction": result['final_answer'],
        "raw_answer": result['raw_answer'],
        "n_clues": len(result.get('all_clues', [])),
        "ranked_sources": ranked_sources,
        "top_clue": result['ranked'][0]['text'][:200] if result.get('ranked') else "",
    }
    row['exact_match'] = exact_match(row['prediction'], row['gold_answer'])
    row['token_f1']    = token_f1(row['prediction'],   row['gold_answer'])
    solar_rows.append(row)

    print(f"\n  GOLD:  {row['gold_answer']}")
    print(f"  PRED:  {row['prediction']}")
    print(f"  EM: {'OK' if row['exact_match'] else 'MISS'}  F1: {row['token_f1']:.3f}")

save_jsonl(solar_rows, OUTPUT_DIR / "solar_predictions.jsonl")
pd.DataFrame([dict(r, ranked_sources=json.dumps(r['ranked_sources'])) for r in solar_rows]
             ).to_csv(OUTPUT_DIR / "solar_predictions.csv", index=False)

solar_eval = evaluate([{"prediction": r['prediction'], "gold_answer": r['gold_answer']} for r in solar_rows])
print(f"\n{'='*65}")
print(f"  Solar: EM={solar_eval['EM%']}%  F1={solar_eval['F1%']}%  N={solar_eval['n']}")
print(f"{'='*65}")

In [ ]:
# ============================================================
# Cell 16 — Ablation study (Paper Appendix B)
# Paper: 'both global and local textualization have positive effects'
# ============================================================

if RUN_BASELINES:
    ablation_results = {}

    # Ablation A: w/o BERT Ranking (unranked clues)
    print("Ablation A: w/o BERT Ranking...")
    rows_a = []
    for ex in examples:
        clues = build_clue_corpus(ex['text_ids'], ex['table_id'], ex['img_ids'])
        raw  = generate_answer(ex['question'], clues[:N_RANK])
        pred = clean_answer(raw)
        rows_a.append({"prediction": pred, "gold_answer": ex['gold_answer']})
        print(f"  [{ex['type']}] {pred[:50]}")
    ablation_results['w/o Ranking'] = evaluate(rows_a)

    # Ablation B: w/o Local Textualization (global only)
    print("\nAblation B: w/o Local (Global Only)...")
    rows_b = []
    for ex in examples:
        clues = []
        for tid in ex['text_ids']:
            t = text_lookup.get(tid)
            if t: clues.append({"text": f"{t['title']}. {t['text']}", "source": "text", "id": tid})
        if ex['table_id']:
            tbl = table_lookup.get(ex['table_id'])
            if tbl:
                ct = textualize_table(tbl)
                if ct: clues.append({"text": ct, "source": "table", "id": ex['table_id']})
        for iid in ex['img_ids']:
            img = image_lookup.get(iid)
            if not img: continue
            cache = IMAGE_CACHE / f"{iid}.jpg"
            lp = str(cache) if cache.exists() else None
            if lp:
                try:
                    image = Image.open(lp).convert("RGB")
                    gt = global_textualize(image, img['title'])
                    if gt: clues.append({"text": gt, "source": "image", "id": iid})
                except: pass
            elif img['title']:
                clues.append({"text": img['title'], "source": "image", "id": iid})
        top  = rank_clues(ex['question'], clues, N_RANK) if clues else []
        pred = clean_answer(generate_answer(ex['question'], top))
        rows_b.append({"prediction": pred, "gold_answer": ex['gold_answer']})
        print(f"  [{ex['type']}] {pred[:50]}")
    ablation_results['w/o Local Textualization'] = evaluate(rows_b)

    print(f"\n{'='*65}")
    print("  ABLATION (Paper Appendix B)")
    print(f"{'='*65}")
    print(f"{'Method':<30} {'EM%':>6} {'F1%':>6} {'ΔF1':>8}")
    print("─" * 52)
    print(f"{'Solar (full)':<30} {solar_eval['EM%']:>6} {solar_eval['F1%']:>6}")
    for name, res in ablation_results.items():
        delta = res['F1%'] - solar_eval['F1%']
        print(f"{'  '+name:<30} {res['EM%']:>6} {res['F1%']:>6} {delta:>+8.1f}%")
    print("=" * 52)

    abl = [{"method": "Solar (full)", **solar_eval}]
    abl += [{"method": name, **v} for name, v in ablation_results.items()]
    pd.DataFrame(abl).to_csv(OUTPUT_DIR / "ablation_study.csv", index=False)
    print("✅ Saved")
else:
    print("RUN_BASELINES=False")

In [ ]:
# ============================================================
# Cell 17 — Final summary
# ============================================================
print("\n" + "="*65)
print("  CASE STUDY — Solar Reasoning Chain (Paper Figure 1)")
print("="*65)

for row in solar_rows:
    print(f"\n  [{row['type']}] {row['question']}")
    print(f"  Top clue [{row['ranked_sources'][0] if row['ranked_sources'] else '?'}]:")
    print(f"    {row['top_clue'][:150]}")
    src_counts = Counter(row['ranked_sources'])
    print(f"  Ranked sources: {dict(src_counts)}")
    print(f"  Pred: {row['prediction']} | Gold: {row['gold_answer']}")
    print(f"  EM: {'OK' if row['exact_match'] else 'MISS'}  F1: {row['token_f1']:.3f}")

# Per-type results
print(f"\n{'='*65}")
print("  PER-TYPE BREAKDOWN")
print(f"{'='*65}")
type_groups = {}
for row in solar_rows:
    t = row['type'].split('(')[0]
    type_groups.setdefault(t, []).append(row)
print(f"{'Type':<20} {'EM%':>6} {'F1%':>6} {'N':>4}")
print("─" * 38)
for qtype, rows in sorted(type_groups.items()):
    g = evaluate([{"prediction": r['prediction'], "gold_answer": r['gold_answer']} for r in rows])
    print(f"{qtype:<20} {g['EM%']:>6} {g['F1%']:>6} {g['n']:>4}")
print("─" * 38)
print(f"{'TOTAL':<20} {solar_eval['EM%']:>6} {solar_eval['F1%']:>6} {solar_eval['n']:>4}")

print(f"\n{'='*65}")
print("  FINAL SUMMARY")
print(f"{'='*65}")
print(f"  Architecture:  Textualize → Rank → Generate")
print(f"  Table:         Natural language ('Row one's X is Y...')")
print(f"  Image Global:  BLIP caption + title")
print(f"  Image Local:   DETR object detection ({'ON' if USE_LOCAL_TEXTUALIZATION else 'OFF'})")
print(f"  Ranker:        BERT cross-encoder | Top-N={N_RANK}")
print(f"  Generator:     T5-UnifiedQA-Large | Beam=4")
print(f"  Solar here:    EM={solar_eval['EM%']}%  F1={solar_eval['F1%']}%")
print(f"  Paper (full):  EM=69.7%  F1=74.8% (fine-tuned, full dataset)")
print(f"\n  Outputs: {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.glob("*.csv")) + sorted(OUTPUT_DIR.glob("*.jsonl")):
    print(f"    {f.name}  ({f.stat().st_size/1024:.1f} KB)")